# Function Messages

The `function.py` module defines messages used to return the result of an executed function to a chat model.

`FunctionMessage` follows an older message schema than `ToolMessage`. It does not contain a `tool_call_id` field, so it cannot directly associate a function result with a specific tool call when multiple tool calls are generated in parallel.

# FunctionMessage: `BaseMessage`

`FunctionMessage` represents the result of a function execution returned to a chat model.
**Syntax**
  ```python
  FunctionMessage(
    self,
    content: str | list[str | dict[Any, Any]] | None = None,
    content_blocks: list[types.ContentBlock] | None = None,
    **kwargs: Any = {}
  )
  ```
## Fields
1. `name`:`str`:= Stores the name of the function that was executed.
2. `type`:`Literal["function"]`:= Stores the message type used during serialization and deserialization. Its default value is `"function"`.

In [1]:
import json

from langchain_core.messages import FunctionMessage


def add_numbers(first: int, second: int) -> dict:
    """Add two numbers and return the function result."""
    return {
        "first": first,
        "second": second,
        "result": first + second
    }


function_result = add_numbers(
    first=20, # First number
    second=30 # Second number
)


message = FunctionMessage(
    content=json.dumps(function_result), # Function execution result
    name="add_numbers", # Name of the executed function
    id=101, # Numeric ID automatically converted to string
    additional_kwargs={
        "status": "success" # Additional provider-specific data
    },
    response_metadata={
        "execution_time_ms": 2 # Function execution metadata
    }
)


print("Content:", message.content) # Display function result
print("Name:", message.name) # Display executed function name
print("Type:", message.type) # Display serialization type
print("ID:", message.id) # Display message identifier
print("ID type:", type(message.id)) # Confirm ID is stored as string
print("Additional kwargs:", message.additional_kwargs) # Display additional data
print("Response metadata:", message.response_metadata) # Display response metadata
print("Content blocks:", message.content_blocks) # Display standardized content blocks
print("Text:", message.text) # Display extracted text
print("Has tool_call_id:", hasattr(message, "tool_call_id")) # False for FunctionMessage

print("\nPretty representation:")
print(
    message.pretty_repr(
        html=False # Use plain-text formatting
    )
)

print("\nPretty print:")
message.pretty_print() # Print the formatted message

Content: {"first": 20, "second": 30, "result": 50}
Name: add_numbers
Type: function
ID: 101
ID type: <class 'str'>
Additional kwargs: {'status': 'success'}
Response metadata: {'execution_time_ms': 2}
Content blocks: [{'type': 'text', 'text': '{"first": 20, "second": 30, "result": 50}'}]
Text: {"first": 20, "second": 30, "result": 50}
Has tool_call_id: False

Pretty representation:
=============================== Function Message ===============================
Name: add_numbers

{"first": 20, "second": 30, "result": 50}

Pretty print:
=============================== Function Message ===============================
Name: add_numbers

{"first": 20, "second": 30, "result": 50}


# FunctionMessageChunk: `FunctionMessage`, `BaseMessageChunk`
`FunctionMessageChunk` represents a partial function-result message produced during streaming. Compatible chunks can be combined while preserving the function name and merging their content and metadata.
**Syntax**
  ```python
  FunctionMessageChunk(
    self,
    content: str | list[str | dict[Any, Any]] | None = None,
    content_blocks: list[types.ContentBlock] | None = None,
    **kwargs: Any = {}
  )
  ```

## Fields
1. `type`:`Literal["FunctionMessageChunk"]`:= Stores the chunk-specific message type used during serialization and deserialization. Its default value is `"FunctionMessageChunk"`.
## Methods
1. `__add__`:= Combines the current function-message chunk with another compatible message chunk.

   When two `FunctionMessageChunk` objects are combined, their function names must match. Otherwise, a `ValueError` is raised.

   The method merges message content, additional keyword arguments, and response metadata while preserving the current chunk's function name and identifier.

   ```python
   __add__(
       self,
       other: Any # Message chunk or another supported value to combine
   ) -> BaseMessageChunk
   ```


In [3]:
from langchain_core.messages import FunctionMessageChunk
chunk1 = FunctionMessageChunk(
    content='{"result": ', # First partial function result
    name="add_numbers", # Name of the executed function
    id="function-chunk-101", # Identifier preserved after merging
    additional_kwargs={
        "status": "processing" # Provider-specific data
    },
    response_metadata={
        "chunk_number": 1 # Metadata of the first chunk
    }
)

chunk2 = FunctionMessageChunk(
    content='"50"}', # Second partial function result
    name="add_numbers", # Must match the first chunk's function name
    additional_kwargs={
        "format": "json" # Additional provider-specific data
    },
    response_metadata={
        "completed": True # Metadata of the second chunk
    }
)
combined_chunk = chunk1 + chunk2 # Combine compatible function-message chunks
print("Content:", combined_chunk.content) # Display merged function result
print("Name:", combined_chunk.name) # Display preserved function name
print("Type:", combined_chunk.type) # Display chunk-specific message type
print("ID:", combined_chunk.id) # Display preserved identifier
print("Additional kwargs:", combined_chunk.additional_kwargs) # Display merged additional data
print("Response metadata:", combined_chunk.response_metadata) # Display merged metadata
print("Content blocks:", combined_chunk.content_blocks) # Display standardized content blocks
print("Text:", combined_chunk.text) # Display extracted text


try:
    invalid_chunk = FunctionMessageChunk(
        content='"Invalid result"}', # Partial result from another function
        name="subtract_numbers" # Different function name
    )

    result = chunk1 + invalid_chunk # Raises ValueError

except ValueError as error:
    print("ValueError:", error) # Display function-name mismatch error

Content: {"result": "50"}
Name: add_numbers
Type: FunctionMessageChunk
ID: function-chunk-101
Additional kwargs: {'status': 'processing', 'format': 'json'}
Response metadata: {'chunk_number': 1, 'completed': True}
Content blocks: [{'type': 'text', 'text': '{"result": "50"}'}]
Text: {"result": "50"}
ValueError: Cannot concatenate FunctionMessageChunks with different names.
